In [105]:
import pandas as pd
from rapidfuzz import fuzz

col_sei = pd.read_csv(r"C:\Users\luiz.smorais\Documents\STAGE\downloads\andamento_2025-09-04.csv", header = 0, usecols= ["Descrição"] )
col_compras = pd.read_csv(r'C:\Users\luiz.smorais\Documents\STAGE\py aux\tabelas_compras\tabelas_compras.csv', header = 0, usecols= ["DESC_TAREFA"] )

In [106]:
display(col_sei)
display(col_compras)

,Descrição
0,Processo atribuído para janete.botelho
1,Processo recebido na unidade
2,Conclusão automática de processo na unidade
3,Processo remetido pela unidade COAJUD
4,ENCAMINHAMOS OS AUTOS À COORDENAÇÃO DE PLANEJA...
...,...
98477,Processo recebido na unidade
98478,Processo remetido pela unidade CGPJUD
98479,Processo recebido na unidade
98480,Processo remetido pela unidade CGCEAF


,DESC_TAREFA
0,Prosseguimento Inicial (Parecer Técnico)
1,Termo Aditivo (Análise de Viabilidade)
2,Autorização da Contratação Direta
3,Termo Aditivo (Elaboração de Parecer Técnico)
4,ARP (Elaboração de Parecer Técnico)
5,Autorização de Publicação do Edital (IRP
6,Audiência Pública
7,Termo Aditivo (Autorização do Aditivo)
8,Encaminhamentos Diversos
9,ARP (Autorização da Execução)


In [118]:
import pandas as pd
import unicodedata
from difflib import SequenceMatcher
from rapidfuzz import fuzz
import time

# Função para normalizar texto
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto

# Simulação de col_sei e descrições
col_sei = [
    "aquisição de papel A4",
    "compra de resma sulfite",
    "contratação de serviço de limpeza",
    "manutenção de impressoras",
    "fornecimento de água mineral"
]

col_compras = [
    "compra de papel sulfite tamanho A4",
    "serviço de limpeza contratado",
    "manutenção preventiva de impressoras",
    "fornecimento de garrafas de água",
    "aquisição de resmas de papel"
]

# Normalizar
col_sei = [normalizar(d) for d in col_sei]
col_compras = [normalizar(d) for d in col_compras]

# Benchmarking
resultados = []

algoritmos = {
    "Levenshtein": lambda a, b: SequenceMatcher(None, a, b).ratio() * 100,
    "Jaro-Winkler": lambda a, b: fuzz.WRatio(a, b),
    "RapidFuzz (token_set_ratio)": lambda a, b: fuzz.token_set_ratio(a, b)
}

for nome_alg, funcao in algoritmos.items():
    inicio = time.time()
    scores = []
    for d1 in col_sei:
        for d2 in col_compras:
            score = funcao(d1, d2)
            scores.append(score)
    fim = time.time()
    resultados.append({
        "Algoritmo": nome_alg,
        "Score Médio": round(sum(scores)/len(scores), 2),
        "Tempo (s)": round(fim - inicio, 4)
    })

# Exibir resultados
df_resultados = pd.DataFrame(resultados)
print("Benchmarking de algoritmos de fuzzy matching:")
print(df_resultados)


Benchmarking de algoritmos de fuzzy matching:
                     Algoritmo  Score Médio  Tempo (s)
0                  Levenshtein        48.51     0.0026
1                 Jaro-Winkler        60.30     0.0070
2  RapidFuzz (token_set_ratio)        53.81     0.0003
